In [ ]:
# @title Momentum Scanner (MWF Trading Schedule)
# @markdown ### 🚀 Strategy: Aggressive Growth (Time Diversified)
# @markdown 1.  **Scanner:** Top 10 Diversified Heavyweights (Price > $40, ADX > 25).
# @markdown 2.  **Schedule:** Enter 1 trade on **Mon, Wed, Fri** to smooth out market volatility.
# @markdown 3.  **Goal:** 10% Monthly Portfolio Growth via Asymmetric Risk/Reward.

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import os
import pickle
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# --- CONFIGURATION ---
MIN_PRICE = 40.00        
MAX_RESULTS = 10         
CACHE_FILE = "market_data_cache.pkl"

# Master List
tickers_raw = "A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BITO BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIA DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EEM EFA EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EWJ EWW EWY EWZ EXC EXPE F FANG FAST FCX FDX FE FI FITB FOXA FSLR FTI FTV FXE FXI GD GDX GE GILD GLD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HBI HCA HD HIG HLT HOG HOLX HON HPE HPQ HYG IBB IBIT IBM ICE IEF INTC IP IPG IRM IVZ IWM IYR JCI JD JETS JNJ JNPR JPM K KHC KMI KO KR KRE KSS KWEB LEN LI LKQ LLY LNC LOW LQD LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OIH OKE OMC ON ORCL OXY PARA PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM QQQ RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHD SCHW SEDG SHOP SIRI SLB SLV SMCI SMH SNAP SNOW SO SOFI SOXL SOXS SPG SPX SPXL SPXS SPY SQQQ STX SWKS SYF SYY T TAP TBT TCOM TDOC TFC TGT TJX TLT TMO TMUS TPR TQQQ TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNG UNH UNP UPS UPST URBN USB USO UVXY V VALE VFC VGK VTR VXX VZ WAB WBA WBD WDC WFC WM WMB WMT WU WY WYNN X XBI XEL XHB XLB XLC XLE XLF XLI XLK XLP XLRE XLU XLV XLY XOM XOP XRT XRX XSP XYZ YELP YINN ZION ZM"
ticker_list = list(set(tickers_raw.split()))

# --- DATA LOADING ---
def get_market_data(tickers):
    today_str = datetime.date.today().strftime("%Y-%m-%d")
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, 'rb') as f:
                cache = pickle.load(f)
            if cache['data'] is not None and not cache['data'].empty:
                print(f"💾 Cache Found. Loading data...")
                return cache['data']
        except: pass
    
    print(f"⏳ Downloading fresh data...")
    data = yf.download(tickers, period="6mo", group_by='ticker', threads=True, progress=False)
    with open(CACHE_FILE, 'wb') as f: pickle.dump({'date': today_str, 'data': data}, f)
    return data

# --- SECTOR FETCHING ---
def get_sector_data(survivors):
    print(f"🌍 Fetching Sector Data for {len(survivors)} candidates...")
    enriched = []
    for i, item in enumerate(survivors):
        try:
            t = yf.Ticker(item['Symbol'])
            sec = t.info.get('sector', 'Unknown')
            if sec == 'Unknown': sec = "ETF / Fund"
            item['Sector'] = sec
        except:
            item['Sector'] = 'Unknown'
        enriched.append(item)
        if i % 5 == 0: print(f"   Fetched {i}/{len(survivors)}...", end="\r")
    return enriched

# --- MATH ---
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calc_sma(series, period=50):
    return series.rolling(window=period).mean()

def calc_adx_robust(df, period=14):
    local_df = df.copy()
    local_df['H-L'] = local_df['High'] - local_df['Low']
    local_df['H-PC'] = abs(local_df['High'] - local_df['Close'].shift(1))
    local_df['L-PC'] = abs(local_df['Low'] - local_df['Close'].shift(1))
    local_df['TR'] = local_df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
    local_df['+DM'] = np.where((local_df['High'] - local_df['High'].shift(1)) > (local_df['Low'].shift(1) - local_df['Low']), 
                               np.maximum(local_df['High'] - local_df['High'].shift(1), 0), 0)
    local_df['-DM'] = np.where((local_df['Low'].shift(1) - local_df['Low']) > (local_df['High'] - local_df['High'].shift(1)), 
                               np.maximum(local_df['Low'].shift(1) - local_df['Low'], 0), 0)
    local_df['TR_S'] = local_df['TR'].ewm(alpha=1/period, adjust=False).mean()
    local_df['+DM_S'] = local_df['+DM'].ewm(alpha=1/period, adjust=False).mean()
    local_df['-DM_S'] = local_df['-DM'].ewm(alpha=1/period, adjust=False).mean()
    local_df['+DI'] = 100 * (local_df['+DM_S'] / local_df['TR_S'])
    local_df['-DI'] = 100 * (local_df['-DM_S'] / local_df['TR_S'])
    local_df['DX'] = 100 * abs(local_df['+DI'] - local_df['-DI']) / (local_df['+DI'] + local_df['-DI'])
    local_df['ADX'] = local_df['DX'].ewm(alpha=1/period, adjust=False).mean()
    return local_df['ADX']

# --- SCANNER ---
def scan_momentum(data, tickers):
    candidates = []
    print(f"🔍 Scanning: Price > ${MIN_PRICE} | ADX > 25 | Uptrend")
    
    for ticker in tickers:
        try:
            try: df = data[ticker].copy().dropna()
            except: continue
            if len(df) < 50: continue

            df['SMA50'] = calc_sma(df['Close'], 50)
            df['RSI'] = calc_rsi(df['Close'])
            df['ADX'] = calc_adx_robust(df)
            
            close = df['Close'].iloc[-1]
            sma50 = df['SMA50'].iloc[-1]
            rsi = df['RSI'].iloc[-1]
            adx = df['ADX'].iloc[-1]
            
            if np.isnan(sma50) or np.isnan(adx): continue

            # FILTERS
            is_heavyweight = close > MIN_PRICE
            is_uptrend = close > sma50
            is_strong = adx > 25
            is_bullish = 50 < rsi < 75 
            
            if is_heavyweight and is_uptrend and is_strong and is_bullish:
                candidates.append({
                    'Symbol': ticker, 'Price': close, 'ADX': adx, 'RSI': rsi
                })
        except: continue
    return candidates

# --- EXECUTION ---
market_data = get_market_data(ticker_list)
tech_pass = scan_momentum(market_data, ticker_list)

if not tech_pass:
    print("❌ No momentum stocks found.")
else:
    # 1. Fetch Sectors
    enriched_list = get_sector_data(tech_pass)
    
    # 2. Sort by ADX
    sorted_candidates = sorted(enriched_list, key=lambda x: x['ADX'], reverse=True)
    
    # 3. Diversity Algorithm
    final_list = []
    seen_sectors = set()
    skipped = []

    for item in sorted_candidates:
        sec = item['Sector']
        if sec not in seen_sectors:
            final_list.append(item)
            seen_sectors.add(sec)
        else:
            skipped.append(item)
        if len(final_list) == MAX_RESULTS: break
    
    if len(final_list) < MAX_RESULTS:
        needed = MAX_RESULTS - len(final_list)
        final_list.extend(skipped[:needed])
        
    final_list = sorted(final_list, key=lambda x: x['ADX'], reverse=True)

    # OUTPUT
    df_final = pd.DataFrame(final_list)
    df_final.index += 1 
    df_final.index.name = 'Rank'
    df_final = df_final.reset_index()
    
    disp = df_final.copy()
    disp['Price'] = disp['Price'].apply(lambda x: f"${x:.2f}")
    disp['ADX'] = disp['ADX'].apply(lambda x: f"{x:.1f}")
    disp['Sector'] = disp['Sector'].apply(lambda x: (x[:18] + '..') if len(x) > 20 else x)

    print("\n" + "="*80)
    print(f"🚀 TOP {len(df_final)} DIVERSIFIED HEAVYWEIGHTS (>$40)")
    print(f"Sorted by: Trend Strength (ADX)")
    print("="*80)
    print(disp[['Rank', 'Symbol', 'Price', 'ADX', 'Sector']].to_string(index=False))
    
    print("\n" + "="*30 + " TRADINGVIEW LIST " + "="*30)
    print(",".join(df_final['Symbol'].tolist()))

# --- STRATEGY GUIDE ---
print("\n" + "="*80)
print("🚀 WEEKLY EXECUTION PLAN (M-W-F)")
print("="*80)
print(f"""
1. TRADING SCHEDULE (Max Spacing):
   - MONDAY:    Enter Trade #1 (Rank 1 Stock).
   - WEDNESDAY: Enter Trade #2 (Rank 2 Stock).
   - FRIDAY:    Enter Trade #3 (Rank 3 Stock).
   - *Logic: Prevents entering 100% of risk on a single market day.*

2. ALLOCATION ($10k Account):
   - Total Weekly Budget: ~$1,500 - $1,800.
   - Per Trade Risk: $500 (approx 3 contracts).
   - Max Loss per Week: If Stops hit on all 3, loss is ~$750 (7.5% drawdown).
   - Max Gain per Week: If all 3 win, gain is ~$1,500 (15% gain).

3. TRADE SETUP (45 DTE, $5 WIDE):
   - Buy ATM Call / Sell OTM Call (+$5).
   - Debit: ~$1.65 - $1.80.

4. MANAGEMENT (The Edge):
   - TAKE PROFIT: GTC Limit Order @ 100% Return (Double your money).
   - STOP LOSS: If Daily Close < 50 SMA.
   - MATH: 50% Win Rate = massive profit due to 2:1 Reward/Risk.
""")